# O5 — Teacher-forced gold-sequence likelihood (Probe-1 grid)

**Floor-free robustness measure.** Binary retention is undefined at floor and ceiling; the 0.30 accuracy floor suppresses most retention cells; N3 collapsed when W3 accuracy was 1/60 and 0/61. Log-likelihood of the gold answer is continuous, defined everywhere, and deterministic (no sampling noise).

**Models (HuggingFace, Colab T4):**
| Model | Load | Role |
|-------|------|------|
| `Qwen/Qwen2.5-1.5B-Instruct` | fp16, `attn_implementation="sdpa"` | primary |
| `Qwen/Qwen2.5-3B-Instruct` | fp16, `attn_implementation="sdpa"` | primary |
| `meta-llama/Llama-3.1-8B-Instruct` | 4-bit NF4, `compute_dtype=float16`, sdpa | robustness check only |

**T4 hard constraints:** fp16 only (no bf16), no FlashAttention-2, `attn_implementation="sdpa"`.

**Grid:** every `(family ∈ {GSM, ALGO, BW}, problem_id, variant ∈ {canonical, W1…W6}, model)` present in the question banks.

**Prompt:** identical Appendix-N Probe-1 template used by the other Colab Probe-1 notebooks (`PROBE1_TEMPLATE` + `FAMILY_FORMAT` + chat template). Do not rewrite.

**Output:** `colab_out/O5_teacher_forced_likelihood.csv` — **per-item rows only**. No aggregates here (analysis is O10).

**Secrets:** `HF_TOKEN` (gated Llama); optional `GITHUB_TOKEN` if the repo is private.

---

### CRITICAL CAVEAT — gold TARGET STRING can change under W3 / W6

W3 renames entities and W6 regenerates parameters, so the gold **target string** often differs from canonical. A raw Δ mean_logprob then confounds *"the model's belief moved"* with *"we are scoring a different string."*

This notebook handles that as follows:
1. **Primary metric is always `mean_logprob`** (length-normalized). Persist `sum_logprob` but do not treat it as the comparison unit.
2. **`target_identical`:** True iff normalized variant gold == normalized canonical gold (cleanest Δ cells; report separately in O10). Checked for all variants; especially informative for W1/W2/W4/W5.
3. **`target_comparable` + `control_*`:** when the variant gold is well-formed under the **canonical** prompt, also teacher-force that gold under the canonical prompt and persist `control_mean_logprob` (etc.). When impossible, `target_comparable=False` and control fields are empty.
4. Well-formed rule (documented, deterministic): identical golds → comparable; else GSM numeric golds remain format-valid under any GSM prompt → comparable; ALGO/BW with a **different** gold → not comparable (entity/operator/instance mismatch).


In [ ]:
# Colab T4: bitsandbytes for quantized loads. Restart the runtime if
# bitsandbytes was just installed and the kernel has not picked it up.
import sys
import subprocess
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.44",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "pandas",
        "scipy",
        "networkx",
        "tqdm",
        "huggingface_hub",
    ]
)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ── knobs ────────────────────────────────────────────────────────────────
# Set LIMIT to an int for a smoke test (e.g. 2 items per family). None = full run.
LIMIT = None
DRY_RUN = False          # True: skip GPU, write placeholder rows (pipeline check)
RESUME = True

# Private GitHub clone (Colab secret GITHUB_TOKEN, or env). Public clone works
# without a token. If this notebook is already inside the repo, clone is skipped.
REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
GH_TOKEN = _secret("GITHUB_TOKEN")

# Llama-3.1-8B-Instruct is gated: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as _hf_exc:
        print("[setup] huggingface login skipped:", _hf_exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "probes" / "contamination" / "verify.py").is_file() and (
        p / "data" / "problems" / "question_bank_gsm.csv"
    ).is_file()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    colab = Path("/content/retrieval-vs-computation")
    if _looks_like_repo(colab):
        return colab
    return colab

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    cmd = ["git", "clone", "--depth", "1", url, str(REPO_ROOT)]
    subprocess.check_call(cmd)
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), (
    f"Could not find probes/ + question banks under {REPO_ROOT}. "
    "Clone the retrieval-vs-computation repo, or set RVC_REPO_URL / GITHUB_TOKEN."
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = Path("/content/colab_out") if Path("/content").exists() else (REPO_ROOT / "colab_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] OUT_DIR={OUT_DIR}")
print(f"[setup] LIMIT={LIMIT} DRY_RUN={DRY_RUN} RESUME={RESUME}")


## Item queue — full Probe-1 bank grid + clone families

Loads `data/problems/question_bank_{gsm,algo,bw}.csv`. Every bank row is one cell. Clone IDs come from `probes.common.clones` / `bank_clone_audit.csv` (ALGO); GSM/BW use `SINGLETON_{problem_id}`.


In [ ]:
from __future__ import annotations

import csv
import gc
import re
from typing import Any

import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from probes.common.clones import algo_cluster_map

# Same Appendix-N template as llama_greedy_behavioural / mechanistic notebooks.
PROBE1_TEMPLATE = (
    "Solve the following problem exactly and provide only the final answer "
    "in the required output format. Problem: {problem}. Format instruction: "
    "{family_specific_output_format}."
)

FAMILY_FORMAT = {
    "GSM": (
        "Write the final numerical answer on its own line as #### <number>. "
        "No other text after that tag."
    ),
    "ALGO": (
        "Follow the problem's required output format exactly "
        "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
    ),
    "BW": (
        "A numbered list of actions only. Each action must be one of the "
        "permitted operators with their arguments. No explanation."
    ),
}

VARIANTS = ("canonical", "W1", "W2", "W3", "W4", "W5", "W6")

MODELS: list[tuple[str, str]] = [
    ("Qwen/Qwen2.5-1.5B-Instruct", "fp16"),
    ("Qwen/Qwen2.5-3B-Instruct", "fp16"),
    ("meta-llama/Llama-3.1-8B-Instruct", "nf4"),  # robustness check only
]

O5_CSV = OUT_DIR / "O5_teacher_forced_likelihood.csv"

OUT_COLUMNS = [
    "family",
    "problem_id",
    "variant",
    "model",
    "n_gold_tokens",
    "sum_logprob",
    "mean_logprob",
    "gold_first_token_rank",
    "gold_first_token_logprob",
    "prompt_n_tokens",
    "clone_family",
    "target_identical",
    "target_comparable",
    "control_n_gold_tokens",
    "control_sum_logprob",
    "control_mean_logprob",
    "control_gold_first_token_rank",
    "control_gold_first_token_logprob",
]


def _norm_vt(v: str) -> str:
    v = str(v).strip()
    return "canonical" if v.lower() == "canonical" else v.upper()


def _strip_csv_quotes(text: str) -> str:
    s = str(text)
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]
    return s


def norm_gold(text: str) -> str:
    """Whitespace-normalized gold for identity checks (not for tokenization)."""
    lines = [ln.strip() for ln in str(text).replace("\r\n", "\n").split("\n")]
    return "\n".join(ln for ln in lines if ln)


def looks_numeric_gold(text: str) -> bool:
    s = norm_gold(text)
    s = re.sub(r"^####\s*", "", s).replace(",", "").strip()
    if not s:
        return False
    try:
        float(s)
        return True
    except ValueError:
        return bool(re.fullmatch(r"-?\d+(?:\.\d+)?", s))


def build_prompt(problem_text: str, family: str) -> str:
    """Identical Probe-1 user string construction as the behavioural Colab notebooks."""
    return PROBE1_TEMPLATE.format(
        problem=problem_text.strip(),
        family_specific_output_format=FAMILY_FORMAT[family],
    )


def _load_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str).fillna("")
    df["problem_id"] = df["problem_id"].astype(str).str.strip()
    df["variant_type"] = df["variant_type"].map(_norm_vt)
    df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
    df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
    return df


def clone_family_for(family: str, problem_id: str, cmap: dict[str, str]) -> str:
    if family == "ALGO":
        return cmap.get(problem_id, f"SINGLETON_{problem_id}")
    return f"SINGLETON_{problem_id}"


def target_flags(
    family: str,
    variant: str,
    can_gold: str,
    var_gold: str,
) -> tuple[bool, bool]:
    """Return (target_identical, target_comparable).

    Comparable ⇒ we may teacher-force *variant gold* under the *canonical* prompt.
    """
    identical = norm_gold(can_gold) == norm_gold(var_gold)
    if identical:
        return True, True
    # Different string: only GSM keeps a format-valid numeric answer under can-prompt.
    if family == "GSM" and looks_numeric_gold(var_gold):
        return False, True
    # W3/W6 (and non-identical ALGO/BW W*) change entities/operators/instance.
    _ = variant  # retained for callers / future tightening
    return False, False


def load_items(limit: int | None) -> list[dict[str, Any]]:
    specs = [
        ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
        ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
        ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
    ]
    cmap = algo_cluster_map()
    items: list[dict[str, Any]] = []
    for family, path in specs:
        df = _load_bank(path)
        can = {
            str(r.problem_id): str(r.correct_answer)
            for r in df.loc[df.variant_type == "canonical"].itertuples(index=False)
        }
        can_text = {
            str(r.problem_id): str(r.problem_text)
            for r in df.loc[df.variant_type == "canonical"].itertuples(index=False)
        }
        for _, row in df.iterrows():
            pid = str(row["problem_id"])
            vt = str(row["variant_type"])
            if vt not in VARIANTS:
                continue
            if pid not in can:
                continue
            var_gold = str(row["correct_answer"])
            c_gold = can[pid]
            identical, comparable = target_flags(family, vt, c_gold, var_gold)
            items.append(
                {
                    "family": family,
                    "problem_id": pid,
                    "variant": vt,
                    "problem_text": str(row["problem_text"]),
                    "gold": var_gold,
                    "canonical_problem_text": can_text[pid],
                    "canonical_gold": c_gold,
                    "target_identical": identical,
                    "target_comparable": comparable and vt != "canonical",
                    "clone_family": clone_family_for(family, pid, cmap),
                }
            )
    if limit is not None:
        # Keep a balanced smoke slice: first `limit` IDs per family × all their variants.
        keep: set[tuple[str, str]] = set()
        for fam in ("GSM", "ALGO", "BW"):
            ids = sorted({x["problem_id"] for x in items if x["family"] == fam})[:limit]
            keep |= {(fam, pid) for pid in ids}
        items = [x for x in items if (x["family"], x["problem_id"]) in keep]
    return items


ITEMS = load_items(LIMIT)
print(f"[queue] {len(ITEMS)} cells (LIMIT={LIMIT})")
print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())
print(
    "[caveat] target_identical rates by variant:\n",
    pd.DataFrame(ITEMS)
    .groupby("variant")["target_identical"]
    .mean()
    .reindex(list(VARIANTS))
    .round(3)
    .to_string(),
)
print(
    "[caveat] target_comparable rates by variant:\n",
    pd.DataFrame(ITEMS)
    .groupby("variant")["target_comparable"]
    .mean()
    .reindex(list(VARIANTS))
    .round(3)
    .to_string(),
)


## Teacher-forced likelihood

For each cell: chat-wrap the Probe-1 user prompt, append the bank gold string (verbatim `correct_answer`), run one forward pass, and sum token log-probs of the gold continuation.

Primary fields: `sum_logprob`, `mean_logprob = sum / n_gold_tokens`, `gold_first_token_rank`, `gold_first_token_logprob`.

When `target_comparable`, also score **variant gold under the canonical prompt** → `control_*` columns.


In [ ]:
def wrap_chat(tokenizer, user_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
        tokenize=False,
    )


def resolve_continuation(
    tokenizer,
    prompt: str,
    answer: str,
) -> tuple[list[int], list[int], str]:
    """Prompt-aware gold token ids (joint encode; try '' then ' ' separator)."""

    def enc(text: str) -> list[int]:
        return tokenizer.encode(text, add_special_tokens=False)

    prompt_ids = enc(prompt)
    answer = str(answer)
    if not answer:
        return prompt_ids, [], "EMPTY"
    candidates: list[tuple[str, list[int], int]] = []
    for sep in ("", " "):
        joint = enc(prompt + sep + answer)
        if len(joint) <= len(prompt_ids):
            continue
        if joint[: len(prompt_ids)] != prompt_ids:
            continue
        rest = joint[len(prompt_ids) :]
        candidates.append((sep, rest, len(joint)))
    if not candidates:
        bare = enc(answer)
        return prompt_ids, bare, "FALLBACK"
    candidates.sort(key=lambda c: c[2])
    sep, rest, _ = candidates[0]
    return prompt_ids, rest, repr(sep)


@torch.inference_mode()
def teacher_forced_metrics(
    model,
    tokenizer,
    device,
    user_text: str,
    gold_text: str,
) -> dict[str, Any]:
    prompt = wrap_chat(tokenizer, user_text)
    prompt_ids, gold_ids, sep_note = resolve_continuation(tokenizer, prompt, gold_text)
    n_prompt = len(prompt_ids)
    n_gold = len(gold_ids)
    if n_gold == 0:
        return {
            "n_gold_tokens": 0,
            "sum_logprob": float("nan"),
            "mean_logprob": float("nan"),
            "gold_first_token_rank": -1,
            "gold_first_token_logprob": float("nan"),
            "prompt_n_tokens": n_prompt,
            "sep_note": sep_note,
        }
    if DRY_RUN or model is None:
        return {
            "n_gold_tokens": n_gold,
            "sum_logprob": 0.0,
            "mean_logprob": 0.0,
            "gold_first_token_rank": 1,
            "gold_first_token_logprob": 0.0,
            "prompt_n_tokens": n_prompt,
            "sep_note": "DRY_RUN",
        }

    input_ids = torch.tensor([prompt_ids + gold_ids], dtype=torch.long, device=device)
    out = model(input_ids=input_ids, use_cache=False)
    # logits[t] predicts token t+1
    logits = out.logits[0]  # [seq, vocab]
    # gold token at absolute index n_prompt + i is predicted by position n_prompt + i - 1
    gold_logits = logits[n_prompt - 1 : n_prompt + n_gold - 1]
    log_probs = F.log_softmax(gold_logits.float(), dim=-1)
    gold_t = torch.tensor(gold_ids, device=device, dtype=torch.long)
    tok_lp = log_probs.gather(1, gold_t.unsqueeze(1)).squeeze(1)
    sum_lp = float(tok_lp.sum().item())
    mean_lp = sum_lp / n_gold

    first_logits = gold_logits[0].float()
    first_tid = int(gold_ids[0])
    first_lp = float(F.log_softmax(first_logits, dim=-1)[first_tid].item())
    rank = int((first_logits > first_logits[first_tid]).sum().item()) + 1

    del out, logits, input_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "n_gold_tokens": n_gold,
        "sum_logprob": round(sum_lp, 6),
        "mean_logprob": round(mean_lp, 6),
        "gold_first_token_rank": rank,
        "gold_first_token_logprob": round(first_lp, 6),
        "prompt_n_tokens": n_prompt,
        "sep_note": sep_note,
    }


def load_model(model_id: str, quant: str):
    assert torch.cuda.is_available() or DRY_RUN, "GPU required (Colab T4) unless DRY_RUN."
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    if DRY_RUN:
        print(f"[model] DRY_RUN skip load: {model_id} ({quant})")
        return tok, None, torch.device("cpu")

    common = dict(
        device_map="auto",
        token=HF_TOKEN or True,
        attn_implementation="sdpa",  # T4: no FlashAttention-2
        torch_dtype=torch.float16,  # T4: fp16 only, no bf16
    )
    if quant == "fp16":
        mdl = AutoModelForCausalLM.from_pretrained(model_id, **common)
        label = "fp16 unquantized + sdpa"
    elif quant == "nf4":
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb,
            **common,
        )
        label = "nf4 4-bit bitsandbytes (compute_dtype=float16) + sdpa"
    else:
        raise ValueError(quant)
    mdl.eval()
    device = next(mdl.parameters()).device
    print(f"[model] {model_id}  {label}  device={device}")
    return tok, mdl, device


def unload(mdl):
    if mdl is None:
        return
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def empty_control() -> dict[str, Any]:
    return {
        "control_n_gold_tokens": "",
        "control_sum_logprob": "",
        "control_mean_logprob": "",
        "control_gold_first_token_rank": "",
        "control_gold_first_token_logprob": "",
    }


def score_item(model, tokenizer, device, item: dict, model_id: str) -> dict[str, Any]:
    user = build_prompt(item["problem_text"], item["family"])
    primary = teacher_forced_metrics(model, tokenizer, device, user, item["gold"])
    row: dict[str, Any] = {
        "family": item["family"],
        "problem_id": item["problem_id"],
        "variant": item["variant"],
        "model": model_id,
        "n_gold_tokens": primary["n_gold_tokens"],
        "sum_logprob": primary["sum_logprob"],
        "mean_logprob": primary["mean_logprob"],
        "gold_first_token_rank": primary["gold_first_token_rank"],
        "gold_first_token_logprob": primary["gold_first_token_logprob"],
        "prompt_n_tokens": primary["prompt_n_tokens"],
        "clone_family": item["clone_family"],
        "target_identical": bool(item["target_identical"]),
        "target_comparable": bool(item["target_comparable"]),
    }
    row.update(empty_control())
    if item["target_comparable"]:
        can_user = build_prompt(item["canonical_problem_text"], item["family"])
        # Control: W-variant gold under the canonical prompt (same gold string, different surface).
        ctrl = teacher_forced_metrics(model, tokenizer, device, can_user, item["gold"])
        row["control_n_gold_tokens"] = ctrl["n_gold_tokens"]
        row["control_sum_logprob"] = ctrl["sum_logprob"]
        row["control_mean_logprob"] = ctrl["mean_logprob"]
        row["control_gold_first_token_rank"] = ctrl["gold_first_token_rank"]
        row["control_gold_first_token_logprob"] = ctrl["gold_first_token_logprob"]
    return row


## Run — write every per-item row (resume-safe)

No aggregate statistics. Resume key: `(model, family, problem_id, variant)`.


In [ ]:
def _done_keys(path: Path) -> set[tuple[str, str, str, str]]:
    if not path.exists():
        return set()
    df = pd.read_csv(path, dtype=str)
    need = {"model", "family", "problem_id", "variant"}
    if not need.issubset(df.columns):
        return set()
    return {
        (str(r.model), str(r.family), str(r.problem_id), str(r.variant))
        for r in df.itertuples(index=False)
    }


def append_rows(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    write_header = not path.exists()
    with path.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=OUT_COLUMNS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in OUT_COLUMNS})


done = _done_keys(O5_CSV) if RESUME else set()
print(f"[resume] {len(done)} rows already in {O5_CSV}")

for model_id, quant in MODELS:
    pending = [
        it
        for it in ITEMS
        if (model_id, it["family"], it["problem_id"], it["variant"]) not in done
    ]
    print(f"\n=== {model_id} ({quant})  pending={len(pending)}/{len(ITEMS)} ===")
    if not pending:
        continue
    tok, mdl, device = load_model(model_id, quant)
    buf: list[dict[str, Any]] = []
    try:
        for it in tqdm(pending, desc=model_id.split("/")[-1]):
            row = score_item(mdl, tok, device, it, model_id)
            buf.append(row)
            if len(buf) >= 25:
                append_rows(O5_CSV, buf)
                buf.clear()
        append_rows(O5_CSV, buf)
    finally:
        unload(mdl)

print(f"\n[done] wrote {O5_CSV}")
if O5_CSV.exists():
    out_df = pd.read_csv(O5_CSV)
    print(f"[done] n_rows={len(out_df)}  models={sorted(out_df['model'].unique())}")
    print(out_df.groupby(["model", "family"]).size().unstack(fill_value=0).to_string())
    # Intentionally no mean/retention aggregates — O10 owns analysis.


## Download / Drive backup

Copy `O5_teacher_forced_likelihood.csv` into the repo as `results/raw/O5_teacher_forced_likelihood.csv` after the Colab run.


In [ ]:
_out_files = [O5_CSV]
_drive_dir = Path("/content/drive/MyDrive/rvc_colab_out")
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)
try:
    import shutil as _shutil
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for p in _out_files:
        if p.exists():
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
except Exception as exc:
    print("[backup] skipped:", exc)
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in _out_files:
        if p.exists():
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
except Exception as exc:
    print("[download] skipped (not Colab or download blocked):", exc)
